In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
ann = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

task_specs = {
    "hatespeech": [0, 1, 2],
    "insult": [0, 1, 2, 3, 4],
    "dehumanize": [0, 1, 2, 3, 4],
    "violence": [0, 1, 2, 3, 4],
    "genocide": [0, 1, 2, 3, 4],
}

In [3]:
raw_rows = []

for_task = []

for task, labels in task_specs.items():
    counts = ann[task].value_counts(normalize=False).sort_index()
    percents = ann[task].value_counts(normalize=True).sort_index() * 100

    for label in labels:
        raw_rows.append({
            "task": task,
            "label": label,
            "annotation_count": counts.get(label, 0),
            "annotation_percent": round(percents.get(label, 0), 2)
        })

raw_label_distribution = pd.DataFrame(raw_rows)
raw_label_distribution

,task,label,annotation_count,annotation_percent
0,hatespeech,0,29951,60.59
1,hatespeech,1,3582,7.25
2,hatespeech,2,15900,32.16
3,insult,0,5399,10.92
4,insult,1,4817,9.74
5,insult,2,6317,12.78
6,insult,3,15877,32.12
7,insult,4,17023,34.44
8,dehumanize,0,10125,20.48
9,dehumanize,1,9854,19.93


In [4]:
majority_rows = []

for task, labels in task_specs.items():
    majority = (
        ann.groupby("comment_id")[task]
        .agg(lambda x: x.value_counts().idxmax())
    )

    counts = majority.value_counts().sort_index()
    percents = majority.value_counts(normalize=True).sort_index() * 100

    for label in labels:
        majority_rows.append({
            "task": task,
            "label": label,
            "majority_comment_count": counts.get(label, 0),
            "majority_comment_percent": round(percents.get(label, 0), 2)
        })

majority_distribution = pd.DataFrame(majority_rows)
majority_distribution

,task,label,majority_comment_count,majority_comment_percent
0,hatespeech,0,13332,67.47
1,hatespeech,1,1462,7.40
2,hatespeech,2,4967,25.14
3,insult,0,2181,11.04
4,insult,1,2054,10.39
5,insult,2,3006,15.21
6,insult,3,7027,35.56
7,insult,4,5493,27.80
8,dehumanize,0,4296,21.74
9,dehumanize,1,4238,21.45


In [5]:
def make_distribution(group, label_col, label_values):
    counts = group[label_col].value_counts(normalize=True)
    return np.array([counts.get(label, 0.0) for label in label_values])

def entropy(probs):
    probs = probs[probs > 0]
    if len(probs) == 0:
        return 0.0
    return -np.sum(probs * np.log2(probs))

entropy_rows = []

for task, labels in task_specs.items():
    task_entropies = []

    for _, group in ann.groupby("comment_id"):
        dist = make_distribution(group, task, labels)
        task_entropies.append(entropy(dist))

    task_entropies = np.array(task_entropies)

    entropy_rows.append({
        "task": task,
        "mean_entropy": task_entropies.mean(),
        "median_entropy": np.median(task_entropies),
        "zero_entropy_percent": round((task_entropies == 0).mean() * 100, 2),
        "max_entropy": task_entropies.max()
    })

entropy_summary = pd.DataFrame(entropy_rows)
entropy_summary

,task,mean_entropy,median_entropy,zero_entropy_percent,max_entropy
0,hatespeech,0.225016,0.0,77.46,1.584963
1,insult,0.380469,0.0,63.17,2.074032
2,dehumanize,0.468168,0.0,57.76,2.238942
3,violence,0.353410,0.0,66.10,2.269211
4,genocide,0.244015,0.0,75.62,2.248174


In [6]:
os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/label_distribution_audit.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("LABEL DISTRIBUTION AUDIT\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. RAW ANNOTATION LABEL DISTRIBUTIONS\n")
    f.write("-" * 80 + "\n")
    f.write(raw_label_distribution.to_string(index=False))
    f.write("\n\n")

    f.write("2. COMMENT-LEVEL MAJORITY LABEL DISTRIBUTIONS\n")
    f.write("-" * 80 + "\n")
    f.write(majority_distribution.to_string(index=False))
    f.write("\n\n")

    f.write("3. COMMENT-LEVEL ENTROPY SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(entropy_summary.to_string(index=False))
    f.write("\n\n")

print(open(report_path, encoding="utf-8").read())

LABEL DISTRIBUTION AUDIT

1. RAW ANNOTATION LABEL DISTRIBUTIONS
--------------------------------------------------------------------------------
      task  label  annotation_count  annotation_percent
hatespeech      0             29951               60.59
hatespeech      1              3582                7.25
hatespeech      2             15900               32.16
    insult      0              5399               10.92
    insult      1              4817                9.74
    insult      2              6317               12.78
    insult      3             15877               32.12
    insult      4             17023               34.44
dehumanize      0             10125               20.48
dehumanize      1              9854               19.93
dehumanize      2              8928               18.06
dehumanize      3             12135               24.55
dehumanize      4              8391               16.97
  violence      0             23484               47.51
  violence     

In [7]:
# ============================================================
# ANNOTATION-LEVEL DEMOGRAPHIC COUNTS
# ============================================================

print("Total annotation rows:", len(ann))

print("\nGender annotation counts:")
display(
    ann["annotator_gender_group"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "gender_group", "annotator_gender_group": "annotation_rows"})
)

print("\nIdeology annotation counts:")
display(
    ann["annotator_ideology_group"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "ideology_group", "annotator_ideology_group": "annotation_rows"})
)

Total annotation rows: 49433

Gender annotation counts:


,annotation_rows,count
0,women,27716
1,men,21112
2,non_binary,362
3,prefer_not_to_say,189
4,self_describe,54



Ideology annotation counts:


,annotation_rows,count
0,liberal,12437
1,neutral,8222
2,slightly_liberal,7844
3,extremely_liberal,6667
4,conservative,5772
5,slightly_conservative,5418
6,extremely_conservative,1655
7,no_opinion,1401
8,unknown,17
